In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 84 — Customer Onboarding Agent

## Goal
Build an agent that **collects required documents** from a new customer,
**verifies completeness**, and **generates a next-step checklist**.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Form-like state** | Collect structured info step by step |
| **Conditional routing** | Branch based on completeness |
| **Checklist generation** | LLM builds personalized next steps |
| **LangGraph** | Conditional edges in state graph |

## Stack
- `langgraph` — stateful graph with conditional edges
- `langchain-ollama` — LLM
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Onboarding Requirements

Define what documents and information are needed
for different customer types.

In [ ]:
# Onboarding requirements by customer type
REQUIREMENTS = {
    "individual": {
        "required_docs": ["government_id", "proof_of_address", "signed_agreement"],
        "required_info": ["full_name", "email", "phone", "date_of_birth"],
        "optional_docs": ["tax_id"],
    },
    "business": {
        "required_docs": ["certificate_of_incorporation", "business_license", "tax_registration", "signed_agreement", "authorized_signatory_id"],
        "required_info": ["company_name", "registration_number", "primary_contact_name", "primary_contact_email", "business_address"],
        "optional_docs": ["financial_statements", "board_resolution"],
    },
}

# Simulated customer submissions
CUSTOMER_SUBMISSIONS = [
    {
        "customer_name": "Alice Johnson",
        "customer_type": "individual",
        "submitted_docs": ["government_id", "signed_agreement"],
        "submitted_info": {
            "full_name": "Alice Johnson",
            "email": "alice@email.com",
            "phone": "+1-555-0101",
            "date_of_birth": ""
        }
    },
    {
        "customer_name": "TechFlow Inc.",
        "customer_type": "business",
        "submitted_docs": ["certificate_of_incorporation", "business_license", "signed_agreement", "authorized_signatory_id", "financial_statements"],
        "submitted_info": {
            "company_name": "TechFlow Inc.",
            "registration_number": "DEL-2024-78901",
            "primary_contact_name": "Bob Chen",
            "primary_contact_email": "bob@techflow.com",
            "business_address": "123 Innovation Blvd, Austin, TX 78701"
        }
    },
]

print(f"Requirements defined for {len(REQUIREMENTS)} customer types")
print(f"Processing {len(CUSTOMER_SUBMISSIONS)} customer submissions")

## Step 2 — LangGraph Onboarding Workflow

```
validate_submission → [complete?] → YES → generate_checklist → END
                                  → NO  → request_missing → generate_checklist → END
```

In [ ]:
class OnboardingState(TypedDict):
    customer: dict
    customer_type: str
    missing_docs: list
    missing_info: list
    extra_docs: list
    is_complete: bool
    validation_notes: list
    missing_items_message: str
    checklist: str

# Node 1: Validate submission against requirements
def validate_submission(state: OnboardingState) -> OnboardingState:
    customer = state["customer"]
    ctype = customer["customer_type"]
    reqs = REQUIREMENTS[ctype]
    
    print(f"\n📋 Validating {customer['customer_name']} ({ctype})...")
    
    # Check documents
    submitted_docs = set(customer.get("submitted_docs", []))
    required_docs = set(reqs["required_docs"])
    optional_docs = set(reqs.get("optional_docs", []))
    
    missing_docs = list(required_docs - submitted_docs)
    extra_docs = list(submitted_docs & optional_docs)
    
    # Check info fields
    submitted_info = customer.get("submitted_info", {})
    missing_info = [f for f in reqs["required_info"] if not submitted_info.get(f)]
    
    # Notes
    notes = []
    if not missing_docs and not missing_info:
        notes.append("All required documents and information received.")
    else:
        if missing_docs:
            notes.append(f"Missing {len(missing_docs)} required document(s).")
        if missing_info:
            notes.append(f"Missing {len(missing_info)} required field(s).")
    if extra_docs:
        notes.append(f"Bonus: {len(extra_docs)} optional document(s) submitted.")
    
    is_complete = len(missing_docs) == 0 and len(missing_info) == 0
    
    print(f"  Docs: {len(submitted_docs)}/{len(required_docs)} required")
    print(f"  Info: {len(reqs['required_info']) - len(missing_info)}/{len(reqs['required_info'])} fields")
    print(f"  Status: {'✅ COMPLETE' if is_complete else '❌ INCOMPLETE'}")
    
    return {
        **state,
        "customer_type": ctype,
        "missing_docs": missing_docs,
        "missing_info": missing_info,
        "extra_docs": extra_docs,
        "is_complete": is_complete,
        "validation_notes": notes,
    }

print("Node: validate_submission defined")

In [ ]:
# Routing function
def route_completeness(state: OnboardingState) -> str:
    return "generate_checklist" if state["is_complete"] else "request_missing"

# Node 2a: Generate request for missing items
def request_missing(state: OnboardingState) -> OnboardingState:
    print("\n📧 Generating missing items request...")
    
    customer = state["customer"]
    
    msg = llm.invoke([
        SystemMessage(content="""Write a polite, professional email to a customer requesting
missing onboarding documents and information. Be specific about what's needed
and provide a deadline of 5 business days. Keep it concise. No thinking."""),
        HumanMessage(content=f"""Customer: {customer['customer_name']}
Type: {state['customer_type']}
Missing documents: {', '.join(state['missing_docs']) if state['missing_docs'] else 'None'}
Missing information: {', '.join(state['missing_info']) if state['missing_info'] else 'None'}""")
    ])
    
    email = msg.content
    if "<think>" in email:
        email = email.split("</think>")[-1].strip()
    
    print(f"  Email drafted ({len(email)} chars)")
    return {**state, "missing_items_message": email}

# Node 3: Generate onboarding checklist
def generate_checklist(state: OnboardingState) -> OnboardingState:
    print("\n✅ Generating onboarding checklist...")
    
    customer = state["customer"]
    
    msg = llm.invoke([
        SystemMessage(content="""Generate a personalized onboarding checklist for this customer.
Include:
1. Document verification status
2. Account setup steps
3. Training/orientation schedule
4. Key contacts to introduce
5. First 30-day milestones
Format as a numbered checklist. Be concise. No thinking."""),
        HumanMessage(content=f"""Customer: {customer['customer_name']}
Type: {state['customer_type']}
Documents complete: {state['is_complete']}
Missing docs: {state['missing_docs']}
Validation notes: {state['validation_notes']}""")
    ])
    
    checklist = msg.content
    if "<think>" in checklist:
        checklist = checklist.split("</think>")[-1].strip()
    
    return {**state, "checklist": checklist}

print("Nodes 2a, 3 defined")

In [ ]:
# Build the graph
graph = StateGraph(OnboardingState)

graph.add_node("validate", validate_submission)
graph.add_node("request_missing", request_missing)
graph.add_node("generate_checklist", generate_checklist)

graph.set_entry_point("validate")
graph.add_conditional_edges("validate", route_completeness, {
    "generate_checklist": "generate_checklist",
    "request_missing": "request_missing"
})
graph.add_edge("request_missing", "generate_checklist")
graph.add_edge("generate_checklist", END)

onboarding_app = graph.compile()
print("Onboarding graph compiled!")

In [ ]:
# Process each customer
for customer in CUSTOMER_SUBMISSIONS:
    print("\n" + "=" * 70)
    print(f"PROCESSING: {customer['customer_name']}")
    print("=" * 70)
    
    result = onboarding_app.invoke({
        "customer": customer,
        "customer_type": "",
        "missing_docs": [],
        "missing_info": [],
        "extra_docs": [],
        "is_complete": False,
        "validation_notes": [],
        "missing_items_message": "",
        "checklist": ""
    })
    
    if result["missing_items_message"]:
        print("\n📧 MISSING ITEMS EMAIL:")
        print(result["missing_items_message"][:500])
    
    print("\n📋 ONBOARDING CHECKLIST:")
    print(result["checklist"][:600])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Document validation** | Compare submitted vs required docs |
| **Conditional routing** | `add_conditional_edges` based on completeness |
| **Missing item request** | LLM drafts professional follow-up email |
| **Checklist generation** | LLM creates personalized next steps |

## Architecture
```
Customer Submission → Validate
                        ↓
              ┌── Complete? ──┐
              │ NO            │ YES
              ↓               ↓
       Request Missing  Generate Checklist
              │               ↑
              └───────────────┘
                        ↓
                       END
```

## Extending This Project
- Add document upload and OCR verification
- KYC/AML screening integration
- Progress dashboard for customer and internal team
- Automated reminders for pending items